# 00 — Configuration and Spark Schema

This notebook loads the YAML configuration, validates the expected file columns, and builds a PySpark `StructType` schema.

**Flow:** Load YAML → Validation of file columns → Extract validated columns → Extract Expected columns → Build Spark schema.

## 1. Imports

In [1]:
from pathlib import Path
from typing import Any
import re

import yaml
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

## 2. Load YAML configuration

In [2]:
def load_yaml_config(yaml_file: str | Path) -> dict[str, Any]:
    """
    Load configuration settings from a YAML file.

    Args:
        yaml_file: Path to the YAML configuration file.

    Returns:
        The configuration loaded from the YAML file.
    Raises:
        FileNotFoundError: If the configuration file does not exist.
        ValueError: If the YAML file is empty.
    """
    config_path = Path(yaml_file)
    
    #Raise error if YAML file not exists
    if not config_path.is_file():
        raise FileNotFoundError("Configuration file does not exist")
    
    # Open the file and safely parse its YAML content.
    try:
        with config_path.open("r", encoding="utf-8") as file:
            settings = yaml.safe_load(file)
    
            # Ensure the YAML is not empty
            if settings is None:
                raise ValueError("YAML should not be empty")
    
            if not isinstance(settings, dict):
                raise TypeError("YAML configuration must contain a dictionary at its root")
        return settings
        
    except yaml.YAMLError as error:
        raise ValueError(f"Invalid YAML configuration syntax:{error}") from error


In [3]:
# test with proper syntax
try:
    conf = load_yaml_config("../config/ingestion_config.yaml")
    print(conf)
except ValueError as error:
    print(f"Configuration Error: {error}")

{'paths': {'target_directory': 'data/raw_files', 'processed_directory': 'data/processed', 'rejected_directory': 'data/rejected', 'log_directory': 'logs'}, 'file_validation': {'file_name_pattern': '^[a-zA-Z]+_[a-zA-Z]+_\\d{8}\\.txt$', 'file_columns': {'id': 'integer', 'name': 'string', 'address': 'string', 'amount': 'double'}}, 'polling': {'interval_minutes': 5, 'execution_budget_minutes': 4}, 'tables': {'audit_table_name': 'ingested_files', 'data_table_name': 'ingested_records'}}


## 3. Define supported Spark data types

In [4]:
# Map the data type names used in YAML to their PySpark equivalents.
TYPE_MAP = {
    "string": StringType(),
    "integer": IntegerType(),
    "double": DoubleType(),
}

# Derive supported names.
SUPPORTED_TYPES = frozenset(TYPE_MAP)

## 4. Validate file column configuration

In [5]:
def validate_file_columns(file_columns: dict[str, str]) -> dict[str, str]:
    """
    Validate and normalize expected file column names and data types.

    Args:
        file_columns: Dictionary mapping column names to type names.

    Returns:
        A new dictionary with trimmed column names and normalized data types.

    Raises:
        TypeError: If the input is not a dictionary, or a key/value is not a string.
        ValueError: If the dictionary is empty, a name/type is empty, a normalized
            column name is duplicated, or a data type is unsupported.
    """
    # Confirm that the supplied configuration has the expected structure.
    if not isinstance(file_columns, dict):
        raise TypeError("file_columns must be a dictionary")

    if not file_columns:
        raise ValueError("file_columns must not be empty")

    validated_columns = {}

    # Validate and normalize each column definition.
    for column_name, type_name in file_columns.items():
        # String methods are only safe after checking both values.
        if not isinstance(column_name, str) or not isinstance(type_name, str):
            raise TypeError("Column names and data types must be strings")

        clean_name = column_name.strip()
        clean_type = type_name.strip().lower()

        if not clean_name:
            raise ValueError("Column name cannot be empty")

        if not clean_type:
            raise ValueError(
                f"Data type cannot be empty for column {column_name!r}"
            )

        # Prevent names such as "id" and " id " becoming duplicate columns.
        if clean_name in validated_columns:
            raise ValueError(
                f"Duplicate column name after trimming: {clean_name!r}"
            )

        if clean_type not in SUPPORTED_TYPES:
            raise ValueError(
                f"Unsupported data type {clean_type!r} for column {clean_name!r}. "
                f"Supported types: {sorted(SUPPORTED_TYPES)}"
            )

        validated_columns[clean_name] = clean_type

    return validated_columns

## 5. Test the column validator

In [6]:
# Positive test: names and data types are normalized.
valid_columns = {" id ": " Integer ", "amount": "DOUBLE"}
validated_columns = validate_file_columns(valid_columns)

print(validated_columns)


{'id': 'integer', 'amount': 'double'}


In [7]:
# Negative tests: each invalid input should raise the expected exception.
invalid_cases = [
    ([], TypeError),                                  # Not a dictionary
    ({}, ValueError),                                 # Empty dictionary
    ({"   ": "string"}, ValueError),                  # Empty column name
    ({"id": "   "}, ValueError),                      # Empty data type
    ({"id": "banana"}, ValueError),                   # Unsupported data type
    ({"id": 123}, TypeError),                         # Data type is not a string
    ({"id": "string", " id ": "integer"}, ValueError), # Duplicate after trimming
]

for invalid_case, expected_error in invalid_cases:
    try:
        validate_file_columns(invalid_case)
    except expected_error as error:
        print(f"{type(error).__name__}: {error}")

TypeError: file_columns must be a dictionary
ValueError: file_columns must not be empty
ValueError: Column name cannot be empty
ValueError: Data type cannot be empty for column 'id'
ValueError: Unsupported data type 'banana' for column 'id'. Supported types: ['double', 'integer', 'string']
TypeError: Column names and data types must be strings
ValueError: Duplicate column name after trimming: 'id'


## 6. Extract file columns from YAML

In [8]:
def get_file_columns(config: dict[str, Any]) -> dict[str, str]:
    """
    Extract and validate file columns from the loaded configuration.

    Args:
        config: Configuration dictionary loaded from YAML.

    Returns:
        Validated file column names and data types.

    Raises:
        KeyError: If the required configuration section or key is missing.
        TypeError or ValueError: If the column configuration is invalid.
    """
    try:
        file_columns = config["file_validation"]["file_columns"]
    except KeyError as error:
        raise KeyError(f"Missing configuration key: {error}") from error

    # Keep all column validation rules in the reusable validator.
    return validate_file_columns(file_columns)

## 7. Load the project configuration

The path below assumes the notebook is run from the `notebooks/` directory, as in the original notebook.

In [9]:
CONFIG_PATH = Path("../config/ingestion_config.yaml")

config = load_yaml_config(CONFIG_PATH)
file_columns = get_file_columns(config)

print(file_columns)

{'id': 'integer', 'name': 'string', 'address': 'string', 'amount': 'double'}


# 8.Call expected columns from get file columns

In [10]:
def get_expected_columns(config: dict[str, Any]) -> list[str]:
    """Extract expected columns from validated file columns.
    Args:
        """
    file_columns = get_file_columns(config)
    return list(file_columns)

In [11]:
# test
config = load_yaml_config(CONFIG_PATH)
print(get_expected_columns(config))

['id', 'name', 'address', 'amount']


# 9.Extract file pattern for the ingestion

In [12]:
def get_file_name_pattern(config: dict[str, Any]) -> re.Pattern:
    """
    Extract, validate, and compile the filename pattern from configuration.

    Args:
        config: Configuration dictionary loaded from YAML.

    Returns:
        A compiled regex pattern for filename validation.

    Raises:
        KeyError: If the required configuration key is missing.
        TypeError: If the filename pattern is not a string.
        ValueError: If the pattern is empty or contains invalid regex syntax.
    """

    try:
        # Extract the filename pattern from the YAML configuration.
        pattern = config["file_validation"]["file_name_pattern"]

    # Raise a descriptive error if a required configuration key is missing.
    except KeyError as error:
        raise KeyError(f"Missing configuration key: {error}") from error

    # Ensure the filename pattern is provided as a string.
    if not isinstance(pattern, str):
        raise TypeError("file_name_pattern must be a string")

    # Remove accidental leading and trailing whitespace.
    pattern = pattern.strip()

    # Reject empty or whitespace-only filename patterns.
    if not pattern:
        raise ValueError("file_name_pattern must not be empty")

    try:
        # Compile the regex pattern and return it for reuse.
        return re.compile(pattern)

    # Convert invalid regex syntax into a configuration validation error.
    except re.error as error:
        raise ValueError(f"Invalid file_name_pattern: {error}") from error


In [13]:
# good
print(get_file_name_pattern(config))

# bad cases
bad_configs = [
    {},                                                          # KeyError
    {"file_validation": {"file_name_pattern": 123}},             # TypeError
    {"file_validation": {"file_name_pattern": "   "}},           # ValueError (empty)
    {"file_validation": {"file_name_pattern": "[a-z"}},          # ValueError (bad regex)
]

for bad_config in bad_configs:
    try:
        get_file_name_pattern(bad_config)
    except (KeyError, TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

re.compile('^[a-zA-Z]+_[a-zA-Z]+_\\d{8}\\.txt$')
KeyError: "Missing configuration key: 'file_validation'"
TypeError: file_name_pattern must be a string
ValueError: file_name_pattern must not be empty
ValueError: Invalid file_name_pattern: unterminated character set at position 0


## 10. Build the Spark schema

In [14]:
def build_spark_schema(file_columns: dict[str, str]) -> StructType:
    """
    Build a PySpark schema from validated file column definitions.

    Args:
        file_columns: Validated dictionary mapping column names to type names.

    Returns:
        A PySpark StructType containing the configured fields.
    """
    fields = []

    # Convert each configured data type into a Spark StructField.
    for column_name, type_name in file_columns.items():
        spark_type = TYPE_MAP[type_name]
        fields.append(StructField(column_name, spark_type, nullable=True))

    # StructType represents the complete schema; StructField represents one column.
    return StructType(fields)

## 10. Build and inspect the schema

In [15]:
spark_schema = build_spark_schema(file_columns)

print(f"schema: {spark_schema}")


schema: StructType([StructField('id', IntegerType(), True), StructField('name', StringType(), True), StructField('address', StringType(), True), StructField('amount', DoubleType(), True)])


# 11.Extracts and validates polling configuration values.

In [16]:
def get_polling_settings(config: dict[str, Any]) -> dict[str, int | float]:
    """
    Extract and validate polling interval and execution budget settings.

    Args:
        config: Configuration dictionary loaded from YAML.

    Returns:
        A dictionary containing the validated polling interval and
        execution budget, both in minutes.

    Raises:
        KeyError: If a required configuration key is missing.
        TypeError: If a setting has an invalid data type.
        ValueError: If a setting is non-positive or the execution budget
                    is not less than the polling interval.
    """
    try:
        # Read both settings from the polling section.
        polling = config["polling"]
        interval = polling["interval_minutes"]
        budget = polling["execution_budget_minutes"]

    except KeyError as error:
        raise KeyError(f"Missing configuration key: {error}") from error

    # bool is a subclass of int, so True would otherwise pass as 1.
    if isinstance(interval, bool) or isinstance(budget, bool):
        raise TypeError("Polling settings must not contain boolean values")

    # Decimals are allowed, for example 0.5 minutes.
    if not isinstance(interval, (int, float)):
        raise TypeError("interval_minutes must be a number")

    if not isinstance(budget, (int, float)):
        raise TypeError("execution_budget_minutes must be a number")

    # A zero or negative time limit makes no sense.
    if interval <= 0 or budget <= 0:
        raise ValueError("Polling settings must be greater than zero")

    # Both values are in minutes, so they can be compared directly.
    # The budget must be shorter than the interval, or there is no rest time.
    if budget >= interval:
        raise ValueError(
            f"execution_budget_minutes ({budget}) must be less than "
            f"interval_minutes ({interval})"
        )

    return {
        "interval_minutes": interval,
        "execution_budget_minutes": budget,
    }

In [17]:
# test
get_polling_settings(conf)

{'interval_minutes': 5, 'execution_budget_minutes': 4}